# CIM Assignment 5 - Experiments with Hybrid Video Encoding/Decoding Scheme


**Course**: Multimedia Information Codification  
**Academic Year**: 2025/2026    
**Authors**: Guilherme Rodrigues 202208878, João Oliveira 202205302         
**Date**: May 2026

## Introduction & Project Objectives

The purpose of this assignment is to apply and consolidate theoretical and practical knowledge regarding hybrid transform-based and perceptually-aware media compression. These core techniques form the foundational building blocks of state-of-the-art image and video compression standards. 

## Key Objectives

- Implement a Simplified Hybrid Encoder/Decoder: Develop a functional, baseline video codec utilizing intra-frame (I-frame), predictive (P-frame), and bidirectional (B-frame) compression modes.  

- Integrate Core Video Coding Tools: Build and test independent functions for block-based 2D Discrete Cosine Transform (DCT2), matrix-based quantization, motion estimation, and motion compensation.  

- Analyze Parameter Trade-offs: Systematically evaluate how varying encoding parameters—such as Group of Pictures (GOP) structures, Macroblock (MB) sizes, pixel accuracy, and quantization scale factors—impact the compression rate and visual quality.  

- Performance Evaluation: Quantify performance by measuring the total bit budget against objective perceptual quality metrics, specifically Peak Signal-to-Noise Ratio (PSNR) and Structural Similarity Index (SSIM).

## Importing libraries

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os
from skimage.metrics import structural_similarity as ssim
import heapq
from scipy.ndimage import zoom
from scipy.fftpack import dct, idct

DEBUG = 0 # To make various experiments that may take a lot of time

# Defining the paths and extensions
img_folder_path = './videos/images_car'
extensions = ('.bmp')

## 1. Experiments with Hybrid Video Encoding

In this phase, we compress an incoming video sequence by isolating and processing only the luminance channel to optimize computational and perceptual efficiency. The encoding workflow sequentially chains motion estimation, block-based spatial transformations, and integer-division quantization matrices.  

## 1.1.i) Image Import and Color Space Conversion

- Process: This component handles importing sequentially named image structures (e.g., .bmp frames from the car video sequence) and handles displaying them on screen.  

- Color Space Flexibility: The system handles dynamic conversions from the default RGB space into alternative perceptually separated representations, specifically YCrCb, LAB, or HSV, depending on the specified user selection parameter.  

- Luminance Extraction: Once split, only the luminance component (Y, L, or V) is passed down the pipeline, while color channels are preserved or set aside to assess spatial-temporal complexity efficiently.

In [ ]:
# Create a list of images
image_files = sorted(os.listdir(img_folder_path),key=lambda x: int(x.replace("car", "").replace(".bmp", "")))

# Selection for different color spaces 
clr_space = 0 # 0 for YCrCb; 1 for LAB; 2 for VHS  

# Function responsible for the conversion
def convert(img_path,clr_space = 0):
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    if (clr_space == 0):
        img_converted = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2YCrCb)
        names = 'YCrCb'
    elif (clr_space == 1):
        img_converted = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
        names = 'LAB'
    elif (clr_space == 2):
        img_converted = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
        names = 'HSV'

    return cv2.split(img_converted), names, img_rgb

### Displaying set of images and respective conversion
Each frame is shown alongside its separated channel components to verify correct conversion.

In [ ]:
for filename in image_files:
    path = os.path.join(img_folder_path, filename)
    print(filename)

    channels, titles, original_img = convert(path, clr_space)

    if clr_space == 0 :
        Y  = []
        Cb = []
        Cr = []
        components = [Y,Cb,Cr]
        components_names = ["Y","Cb","Cr"]
    elif clr_space == 1:
        L = []
        A = []
        B = []
        components = [L,A,B]
        components_names = ["L","A","B"]
    elif clr_space == 2:
        H = []
        S = []
        V = []
        components = [H,S,V]
        components_names = ["H","S","V"]

    components = channels
 
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 4, 1)
    plt.imshow(original_img)
    plt.title(f"Original Image: {filename}")
    plt.axis('off')

    for i in range(2,5) :
        plt.subplot(1,4,i)
        plt.imshow(components[i-2])
        plt.title(f"{components_names[i-2]} channel image: {filename}")

    plt.tight_layout()
    plt.show()

## 1.1.ii) 


### Mean Squared Error
MSE is used as the error metric inside block_matching_v2 to find the best-matching reference block for each macroblock.

In [ ]:
def mse(block_a, block_b):
    diff = block_a.astype(np.float32) - block_b.astype(np.float32)
    return np.mean(diff ** 2)

## 1.1.ii) & 1.1.iii) Variable Block-Size Motion Estimation

- Process: This function calculates motion vectors on a Macroblock (MB) basis relative to reference frames. It computes the optimal motion shift using a fast search method.  

- Input Parameters: The function accepts an anchor image block, reference frames (one frame for forward-predictive P-mode or two frames for bidirectional B-mode), an encoding mode selector, target pixel accuracy, and a search window size configuration.  

- Variable Macroblock Support: To adaptively capture varying motion complexities, block dimensions can scale dynamically from a standard 16x16 size down to a granular 4x4 matrix layout.  

- Outputs: For each processed block, the algorithm outputs its calculated coordinate motion vector pair alongside the corresponding residual difference block.


In [ ]:
def block_matching_v2(target_frame, references, mode='P', accuracy=1, window_size=15, block_size=16, min_block_size=4):

    # Ensure block_size doesn't go below the allowed minimum
    current_block_size = max(block_size, min_block_size)
    
    height, width = target_frame.shape
    search_range = window_size * accuracy

    # Initialize output arrays based on the current block size
    motion_vectors = np.zeros((height // current_block_size, width // current_block_size, 2), dtype=float) # Each macro block is represented by a pair (2) of coordinates -> dx,dy
    diff_blocks = np.zeros((height // current_block_size, width // current_block_size, current_block_size, current_block_size), dtype=np.float32) 

    # Upsample frames when sub-pixel accuracy is requested, for both anchor and reference images
    if accuracy > 1:
        target_upscaled = zoom(target_frame.astype(np.float32), accuracy, order=1)
        refs_up = [zoom(r.astype(np.float32), accuracy, order=1) for r in references]
    else:
        target_upscaled = target_frame.astype(np.float32)
        refs_up = [r.astype(np.float32) for r in references]

    bs_up = current_block_size * accuracy

    # --- START OF SCRIPT -----------

    for i in range(0, height - current_block_size + 1, current_block_size):    # Iterating vertically by current_block_size 16 by 16
        for j in range(0, width - current_block_size + 1, current_block_size): # Iterating horizontally by current_block_size

            best_mse = float('inf')
            best_mv = (0, 0)
            best_ref = 0   

            # Extract anchor block at upsampled resolution

            block_up = target_upscaled[(i * accuracy) : ((i + current_block_size) * accuracy), (j * accuracy):((j + current_block_size) * accuracy)] # Extracting the current 16x16 block 

            if (mode == 'B' and len(references) > 1): # Since it's bidirectional it can use both previous and following images
                ref_range = range(2)  
            else:
                ref_range = range(1)

            for ref_idx in ref_range: # Iterating through the image(s)
                ref_up = refs_up[ref_idx] 
                h_up, w_up = ref_up.shape

                for x in range(-search_range, search_range + 1):
                    for y in range(-search_range, search_range + 1): # Same process as before
                        ri, ci = i * accuracy + x, j * accuracy + y

                        if 0 <= ri <= h_up - bs_up and 0 <= ci <= w_up - bs_up: 
                            candidate = ref_up[ri: (ri + bs_up) , ci: (ci + bs_up)]
                            error = mse(block_up, candidate)

                            if error < best_mse:
                                best_mse = error
                                best_mv = (x / accuracy, y / accuracy)
                                best_ref = ref_idx

            motion_vectors[i // current_block_size, j // current_block_size] = best_mv

    # --- END OF SCRIPT -----------

            # Reconstruct the predicted block for the residual
            bx = int(round((i + best_mv[0]) * accuracy))
            by = int(round((j + best_mv[1]) * accuracy))
            ref_up_best = refs_up[best_ref]
            h_up, w_up = ref_up_best.shape

            if 0 <= bx <= h_up - bs_up and 0 <= by <= w_up - bs_up:
                predicted_up = ref_up_best[bx:bx + bs_up, by:by + bs_up]
                if accuracy > 1:
                    predicted = zoom(predicted_up, 1.0 / accuracy, order=1)
                else:
                    predicted = predicted_up
            else:
                predicted = target_frame[i:i + current_block_size, j:j + current_block_size].astype(np.float32)

            # Store the difference (Residual)
            actual_block = target_frame[i:i + current_block_size, j:j + current_block_size].astype(np.float32)
            diff_blocks[i // current_block_size, j // current_block_size] = actual_block - predicted

    return motion_vectors, diff_blocks

## 1.1.iv) Motion Vector Visualization

- Process: To visually inspect and debug the accuracy of our fast-search block matching, this routine overlays the calculated motion vectors onto the frame layout.  

- Objective: Plotting these vectors allows us to observe spatial correlation trends, detect high-motion regions, and visually evaluate tracking efficiency.


In [ ]:
def plot_motion_vectors(anchor_frame, motion_vectors, block_size=16):

    cols, rows, _ = motion_vectors.shape

    cy = np.array([i * block_size + block_size // 2 for i in range(rows)])  # vertical centres
    cx = np.array([j * block_size + block_size // 2 for j in range(cols)])  # horizontal centres

    CX, CY = np.meshgrid(cx, cy)  # shape (rows, cols) each

    DY = motion_vectors[:, :, 0]  # row offsets
    DX = motion_vectors[:, :, 1]  # column offsets

    plt.figure(figsize=(10, 8))
    plt.imshow(anchor_frame, cmap='gray', vmin=0, vmax=255)
    plt.quiver(CX, CY, DX, DY,
               color='red',
               angles='xy',       # angles measured in display space
               scale_units='xy',  # arrow length in pixel units
               scale=1)           # 1 pixel of MV = 1 pixel on screen

    plt.title("Motion Vectors")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

## 1.1.v) 2D Discrete Cosine Transform (DCT2)

- Process: Spatial redundancy within the luminance channel or residual motion-error blocks is minimized by applying a two-dimensional DCT transformation.  

- Execution: This function splits the frame component into distinct 8x8 pixel blocks, invoking successive operations to map spatial intensities into equivalent frequency-domain coefficients.

In [ ]:
def zigzag_order():
    return [
    (0,0), (0,1), (1,0), (2,0), (1,1), (0,2), (0,3), (1,2), (2,1), (3,0), (4,0), (3,1), (2,2), (1,3), (0,4), (0,5),
    (1,4), (2,3), (3,2), (4,1), (5,0), (6,0), (5,1), (4,2), (3,3), (2,4), (1,5), (0,6), (0,7), (1,6), (2,5), (3,4),
    (4,3), (5,2), (6,1), (7,0), (7,1), (6,2), (5,3), (4,4), (3,5), (2,6), (1,7), (2,7), (3,6), (4,5), (5,4), (6,3),
    (7,2), (7,3), (6,4), (5,5), (4,6), (3,7), (4,7), (5,6), (6,5), (7,4), (7,5), (6,6), (5,7), (6,7), (7,6), (7,7)
    ]


def dct_process(image):

    height, width = image.shape # The image should be a multiple of 8
    h_blocks, w_blocks = height//8, width//8
    dct_blocks = np.zeros((h_blocks,w_blocks,8,8))

    for i in range(h_blocks):
        for j in range(w_blocks):
            block = image[i*8:(i+1)*8, j*8:(j+1)*8].astype(float)
            dct_blocks[i, j] = dct(dct(block, norm='ortho'), norm='ortho', axis=1) # The same as doing T.Block.T'. Orthogonalized variant
            
    return dct_blocks, (h_blocks, w_blocks)



def idct_process(dct_blocks, h_blocks, w_blocks):
    recovered = np.zeros((h_blocks*8, w_blocks*8))

    ZIGZAG_ORDER = zigzag_order()

    for i in range(h_blocks):
        for j in range(w_blocks):
            block = dct_blocks[i, j]
            recovered[i*8:(i+1)*8, j*8:(j+1)*8] = idct(idct(block, norm='ortho'), norm='ortho', axis=1)

    return recovered

## 1.1.vi) Matrix-Based Quantization

- Process: This function performs lossy compression via an element-wise integer division of the 8x8 frequency coefficient blocks.  

- MPEG Scaling Principle: The division utilizes scaled quantization matrices. It applies a default MPEG intra-frame matrix when processing original pixel values (I-mode), and a dedicated inter-frame matrix for residual motion error blocks (P and B modes).  

- Control Mechanism: The user-defined quantization scale factor alters the magnitude of these matrices, acting as the primary control slider to balance overall compression ratios against visual distortions.

In [ ]:
# MPEG default quantisation matrices (MPEG-1/2 spec)
QUANT_MATRIX_I = np.array([
    [ 8, 16, 19, 22, 26, 27, 29, 34],
    [16, 16, 22, 24, 27, 29, 34, 37],
    [19, 22, 26, 27, 29, 34, 34, 38],
    [22, 22, 26, 27, 29, 34, 37, 40],
    [22, 26, 27, 29, 32, 35, 40, 48],
    [26, 27, 29, 32, 35, 40, 48, 58],
    [26, 27, 29, 34, 38, 46, 56, 69],
    [27, 29, 35, 38, 46, 56, 69, 83]
], dtype=np.float32) # For I

QUANT_MATRIX_PB = np.array([
    [16, 17, 18, 19, 20, 21, 22, 23],
    [17, 18, 19, 20, 21, 22, 23, 24],
    [18, 19, 20, 21, 22, 23, 24, 25],
    [19, 20, 21, 22, 23, 24, 26, 27],
    [20, 21, 22, 23, 26, 26, 27, 28],
    [21, 22, 23, 24, 26, 27, 28, 30],
    [22, 23, 24, 26, 27, 28, 30, 31],
    [23, 24, 25, 27, 28, 30, 31, 33]
], dtype=np.float32) # For B and P


def quantise_block(block, mode='I', quant_scale=4):

    base_matrix = QUANT_MATRIX_I if mode == 'I' else QUANT_MATRIX_PB
    scaled_matrix = base_matrix * quant_scale

    q_block = (block / scaled_matrix).astype(np.int16)
    return q_block


def dequantise_block(q_block, mode='I', quant_scale=4):
    
    base_matrix = QUANT_MATRIX_I if mode == 'I' else QUANT_MATRIX_PB
    scaled_matrix = base_matrix * quant_scale

    block = q_block.astype(np.float32) * scaled_matrix
    return block

## 1.1. Main

Demonstrates all functions from 1.1 sequentially on the car image sequence, verifying each component in isolation before integration.

In [ ]:
# ── 1.1 Main Demo ─────────────────────────────────────────────────────────────
# Parameters
CLR_SPACE   = 0    # 0=YCrCb, 1=LAB, 2=HSV
BLOCK_SIZE  = 16   # Macro-block size (try 8 or 16)
MIN_BLOCK   = 4    # Minimum block size for block_matching_v2
ACCURACY    = 1    # Sub-pixel accuracy (1 = integer pixel)
WINDOW_SIZE = 7    # Motion search window
QUANT_SCALE = 4    # Quantiser scale factor

# ── 1.1.i — Load and convert the first frame ──────────────────────────────────
print("── 1.1.i  Color space conversion ──")

frame0_path = os.path.join(img_folder_path, image_files[0])
channels, space_name, original_rgb = convert(frame0_path, CLR_SPACE)
Y0  = channels[0].astype(np.float32)   # luminance
Cb0 = channels[1]
Cr0 = channels[2]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(original_rgb);           axes[0].set_title("Original RGB")
axes[1].imshow(Y0,  cmap='gray');       axes[1].set_title(f"Y  ({space_name})")
axes[2].imshow(Cb0, cmap='gray');       axes[2].set_title(f"Cb ({space_name})")
axes[3].imshow(Cr0, cmap='gray');       axes[3].set_title(f"Cr ({space_name})")
for ax in axes: ax.axis('off')
plt.suptitle(f"1.1.i — {image_files[0]}")
plt.tight_layout()
plt.show()

# ── Load second frame for motion estimation ────────────────────────────────────
frame1_path = os.path.join(img_folder_path, image_files[1])
channels1, _, _ = convert(frame1_path, CLR_SPACE)
Y1 = channels1[0].astype(np.float32)

# ── 1.1.ii / 1.1.iii — Motion estimation (block_matching_v2) ──────────────────
print("\n── 1.1.ii/iii  Motion estimation ──")

motion_vectors, diff_blocks = block_matching_v2(
    target_frame  = Y1,
    references    = [Y0],
    mode          = 'P',
    accuracy      = ACCURACY,
    window_size   = WINDOW_SIZE,
    block_size    = BLOCK_SIZE,
    min_block_size= MIN_BLOCK
)

print(f"  Anchor frame  : {image_files[1]}")
print(f"  Reference     : {image_files[0]}")
print(f"  MB grid shape : {motion_vectors.shape[:2]}  ({BLOCK_SIZE}×{BLOCK_SIZE} blocks)")
print(f"  MV range dy   : [{motion_vectors[:,:,0].min():.1f}, {motion_vectors[:,:,0].max():.1f}]")
print(f"  MV range dx   : [{motion_vectors[:,:,1].min():.1f}, {motion_vectors[:,:,1].max():.1f}]")
print(f"  Residual mean : {np.abs(diff_blocks).mean():.2f}")

# Reconstruct full residual image for display
rows_mb, cols_mb = motion_vectors.shape[:2]
residual_img = np.zeros((rows_mb * BLOCK_SIZE, cols_mb * BLOCK_SIZE), dtype=np.float32)
for i in range(rows_mb):
    for j in range(cols_mb):
        residual_img[i*BLOCK_SIZE:(i+1)*BLOCK_SIZE,
                     j*BLOCK_SIZE:(j+1)*BLOCK_SIZE] = diff_blocks[i, j]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(Y0,  cmap='gray', vmin=0, vmax=255); axes[0].set_title(f"Reference  ({image_files[0]})")
axes[1].imshow(Y1,  cmap='gray', vmin=0, vmax=255); axes[1].set_title(f"Anchor     ({image_files[1]})")
axes[2].imshow(np.abs(residual_img), cmap='hot');   axes[2].set_title("Residual |anchor − predicted|")
for ax in axes: ax.axis('off')
plt.suptitle("1.1.ii/iii — Motion Estimation")
plt.tight_layout()
plt.show()

# ── 1.1.iv — Plot motion vectors ──────────────────────────────────────────────
print("\n── 1.1.iv  Motion vector plot ──")
plot_motion_vectors(Y1, motion_vectors, block_size=BLOCK_SIZE)

# ── 1.1.v — DCT on the first frame (luminance) ────────────────────────────────
print("\n── 1.1.v  DCT / IDCT ──")

# Crop Y0 to a multiple of 8
H8 = (Y0.shape[0] // 8) * 8
W8 = (Y0.shape[1] // 8) * 8
Y0_crop = Y0[:H8, :W8]

dct_blocks, (hb, wb) = dct_process(Y0_crop)
Y0_reconstructed     = idct_process(dct_blocks, hb, wb)

psnr_val = 10 * np.log10(255**2 / np.mean((Y0_crop - Y0_reconstructed)**2))
print(f"  PSNR after DCT → IDCT (lossless round-trip): {psnr_val:.1f} dB")

# Visualise DCT coefficient magnitudes of the first block
first_dct_block = dct_blocks[0, 0]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(Y0_crop,          cmap='gray', vmin=0, vmax=255)
axes[0].set_title("Original Y")
axes[1].imshow(np.log1p(np.abs(first_dct_block)), cmap='viridis')
axes[1].set_title("DCT block [0,0]  (log scale)")
axes[2].imshow(Y0_reconstructed, cmap='gray', vmin=0, vmax=255)
axes[2].set_title(f"Reconstructed Y  (PSNR {psnr_val:.1f} dB)")
for ax in axes: ax.axis('off')
plt.suptitle("1.1.v — DCT / IDCT round-trip")
plt.tight_layout()
plt.show()

# ── 1.1.vi — Quantisation ─────────────────────────────────────────────────────
print("\n── 1.1.vi  Quantisation / Dequantisation ──")

for qs in [2, 4, 8, 16]:   # test several scale factors
    # Quantise every 8x8 block
    q_blocks = np.array(
        [[quantise_block(dct_blocks[i, j], mode='I', quant_scale=qs)
          for j in range(wb)]
         for i in range(hb)], dtype=np.int16
    )

    # Dequantise
    dq_blocks = np.array(
        [[dequantise_block(q_blocks[i, j], mode='I', quant_scale=qs)
          for j in range(wb)]
         for i in range(hb)], dtype=np.float32
    )

    # Reconstruct image
    Y0_rec = np.clip(idct_process(dq_blocks, hb, wb), 0, 255)

    nonzero_ratio = np.count_nonzero(q_blocks) / q_blocks.size
    psnr_q = 10 * np.log10(255**2 / (np.mean((Y0_crop - Y0_rec)**2) + 1e-10))
    print(f"  QScale={qs:2d}  non-zero coeffs: {nonzero_ratio*100:5.1f}%   PSNR: {psnr_q:.2f} dB")

# Visual comparison across scale factors
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(Y0_crop, cmap='gray', vmin=0, vmax=255)
axes[0].set_title("Original")
axes[0].axis('off')

for ax_idx, qs in enumerate([2, 4, 8, 16], start=1):
    q_blocks = np.array(
        [[quantise_block(dct_blocks[i, j], mode='I', quant_scale=qs)
          for j in range(wb)]
         for i in range(hb)], dtype=np.int16
    )
    dq_blocks = np.array(
        [[dequantise_block(q_blocks[i, j], mode='I', quant_scale=qs)
          for j in range(wb)]
         for i in range(hb)], dtype=np.float32
    )
    Y0_rec = np.clip(idct_process(dq_blocks, hb, wb), 0, 255)
    psnr_q = 10 * np.log10(255**2 / (np.mean((Y0_crop - Y0_rec)**2) + 1e-10))
    axes[ax_idx].imshow(Y0_rec, cmap='gray', vmin=0, vmax=255)
    axes[ax_idx].set_title(f"QScale={qs}\nPSNR {psnr_q:.1f} dB")
    axes[ax_idx].axis('off')

plt.suptitle("1.1.vi — Effect of Quantiser Scale Factor (I-frame)")
plt.tight_layout()
plt.show()

## 1.2) Complete Hybrid Video Encoding Pipeline

- Process: This master workflow imports the raw sequence and manages macroblock partitioning across consecutive frames according to a selected Group of Pictures (GOP) configuration.  

- GOP Configurations: The system orchestrates frame types by taking parameters for total sequence length (N) and the spacing interval between anchors (M), managing structures like IPPP or IBBP.  

- Storage Allocation: The complete serial chain of processing operations is systematically applied across all frame units. The resulting quantized frequency coefficients and coordinate motion vectors are efficiently structured and exported to disk files for compression rate verification.

## Extracting frame type
Given a frame index and GOP parameters N and M, returns whether the frame should be encoded as I, P, or B.

In [ ]:
def get_frame_type(frame_idx, N, M):
    pos = frame_idx % N
    if pos == 0: 
        return 'I'
    elif pos % M == 0: 
        return 'P'
    else:
        return 'B'
    
"""
Example for N = 12 and M = 3
0  1  2  3  4  5  6  7  8  9  10  11  12
I  B  B  P  B  B  P  B  B  P   B   B   I
"""

## Counting number of encoded frame's bits
Estimates compressed size by counting non-zero quantised coefficients (×16 bits each) plus motion vector storage.

In [ ]:
def count_bits(encoded):
    coeff_bits = int(np.count_nonzero(encoded['q_blocks'])) * 16
    mv_bits    = int(encoded['motion_vectors'].size) * 16 \
                 if encoded['motion_vectors'] is not None else 0
                 
    return coeff_bits + mv_bits

## Encoding a frame
Applies the full encoding chain: motion estimation (P/B only) → DCT → quantisation, returning a dict of compressed data.

In [ ]:
def encode_frame(Y, frame_type, ref_frames, block_size, min_block_size,
                 accuracy, window_size, quant_scale):
    """
    Encodes a single luminance frame through the full chain:
    Motion Estimation → DCT → Quantisation.

    Parameters:
        Y             : 2D float32 luminance array.
        frame_type    : 'I', 'P', or 'B'.
        ref_frames    : list of decoded reference Y arrays.
        block_size    : macro-block size for motion estimation.
        min_block_size: minimum block size for block_matching_v2.
        accuracy      : sub-pixel accuracy.
        window_size   : motion search window.
        quant_scale   : quantiser scale factor.

    Returns a dict with q_blocks, motion_vectors, and metadata.
    """

    # Crop to be a multiple of both block_size and 8 (required by DCT)
    unit   = max(block_size, 8)
 
    H_crop = (Y.shape[0] // unit) * unit
    W_crop = (Y.shape[1] // unit) * unit

    result = {'type': frame_type, 'shape': Y.shape, 'motion_vectors': None, 'q_blocks': None}

    if frame_type == 'I':
        # Raw luminance → DCT → Quantise (no motion estimation)
        dct_blocks, (hb, wb) = dct_process(Y)
        q_blocks = np.array([[quantise_block(dct_blocks[i, j], mode='I', quant_scale=quant_scale) for j in range(wb)] for i in range(hb)], dtype=np.int16)

    else:
        # P / B: motion estimation on macro-blocks then DCT + quantise the residual
        motion_vectors, diff_blocks = block_matching_v2(
            Y, ref_frames, mode=frame_type,
            accuracy=accuracy, window_size=window_size,
            block_size=block_size, min_block_size=min_block_size
        )
        result['motion_vectors'] = motion_vectors

        # Reassemble the full residual image from diff_blocks
        rows_mb, cols_mb = motion_vectors.shape[:2]
        residual = np.zeros((rows_mb * block_size, cols_mb * block_size),
                            dtype=np.float32)
        for i in range(rows_mb):
            for j in range(cols_mb):
                residual[i*block_size:(i+1)*block_size,
                         j*block_size:(j+1)*block_size] = diff_blocks[i, j]

        dct_blocks, (hb, wb) = dct_process(residual)
        q_blocks = np.array([[quantise_block(dct_blocks[i, j], mode=frame_type,
                                             quant_scale=quant_scale)
                              for j in range(wb)] for i in range(hb)],
                            dtype=np.int16)

    result['q_blocks'] = q_blocks
    result['hb']       = hb
    result['wb']       = wb
    return result

In [ ]:
def decode_frame(encoded, ref_frames, block_size, quant_scale):

    frame_type = encoded['type']
    hb, wb     = encoded['hb'], encoded['wb']
    q_blocks   = encoded['q_blocks']

    # Inverse quantise every 8x8 block
    dct_rec = np.array([[dequantise_block(q_blocks[i, j], mode=frame_type, quant_scale=quant_scale) for j in range(wb)] for i in range(hb)], dtype=np.float32)

    # Inverse DCT → recovers either raw Y (I) or residual (P/B)
    rec_signal = idct_process(dct_rec, hb, wb)

    if frame_type == 'I':
        return np.clip(rec_signal, 0, 255).astype(np.float32)

    # P / B: add motion-compensated prediction to residual
    motion_vectors = encoded['motion_vectors']
    rows_mb, cols_mb = motion_vectors.shape[:2]
    prediction = np.zeros((rows_mb * block_size, cols_mb * block_size),
                          dtype=np.float32)
    ref = ref_frames[0]   # primary reference

    for i in range(rows_mb):
        for j in range(cols_mb):
            dy, dx = motion_vectors[i, j]
            ry = int(round(i * block_size + dy))
            rx = int(round(j * block_size + dx))
            ry = int(np.clip(ry, 0, ref.shape[0] - block_size))
            rx = int(np.clip(rx, 0, ref.shape[1] - block_size))
            prediction[i*block_size:(i+1)*block_size,
                       j*block_size:(j+1)*block_size] = \
                ref[ry:ry + block_size, rx:rx + block_size]

    H_pred, W_pred = prediction.shape
    decoded_Y = prediction + rec_signal[:H_pred, :W_pred]

    return np.clip(decoded_Y, 0, 255).astype(np.float32)

## Turning video to images
Extracts frames from an mp4 file using cv2.VideoCapture and converts each to the chosen colour space, returning (Y, label) tuples.

In [ ]:
def load_video_frames(video_path, clr_space=0, max_frames=None):
    """
    Loads frames from a video file and converts each to the chosen colour space.
    Returns a list of (Y, filename_label) tuples.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    frames = []
    frame_idx = 0

    while True:
        ret, frame_bgr = cap.read()
        if not ret:
            break
        if max_frames and frame_idx >= max_frames:
            break

        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

        if clr_space == 0:
            converted = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2YCrCb)
        elif clr_space == 1:
            converted = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2LAB)
        elif clr_space == 2:
            converted = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2HSV)

        channels = cv2.split(converted)
        Y = channels[0].astype(np.float32)
        label = f"frame_{frame_idx:04d}"
        frames.append((Y, label))
        frame_idx += 1

    cap.release()
    print(f"  Loaded {len(frames)} frames from '{video_path}'")
    return frames

## 1.2. Main
Single-configuration run of the encoder, used to verify the pipeline before the parameter sweep below.

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
VIDEO_PATH   = './videos/input.mp4'
GOP_N        = 9
GOP_M        = 3
BLOCK_SIZE   = 16
MIN_BLOCK    = 4
ACCURACY     = 1
WINDOW_SIZE  = 7
QUANT_SCALE  = 4
CLR_SPACE    = 0
MAX_FRAMES   = 30     # set None to encode the full video
OUTPUT_DIR   = './compressed'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Load frames from video ─────────────────────────────────────────────────────
video_frames = load_video_frames(VIDEO_PATH, CLR_SPACE, MAX_FRAMES)

print("=" * 65)
print(f" GOP N={GOP_N}  M={GOP_M} | Block={BLOCK_SIZE}px | "
      f"Window={WINDOW_SIZE} | QScale={QUANT_SCALE}")
print(f" Frames to encode: {len(video_frames)}")
print("=" * 65)

ref_frames  = []
total_bits  = 0
encoded_seq = []

for frame_idx, (Y, label) in enumerate(video_frames):

    frame_type = get_frame_type(frame_idx, GOP_N, GOP_M)

    if frame_type == 'I':
        refs = []
    elif frame_type == 'P':
        refs = [ref_frames[-1]] if ref_frames else [Y]
    else:
        prev = ref_frames[-1] if ref_frames else Y
        refs = [prev, prev]

    encoded = encode_frame(Y, frame_type, refs,
                           BLOCK_SIZE, MIN_BLOCK,
                           ACCURACY, WINDOW_SIZE, QUANT_SCALE)
    encoded_seq.append((encoded, label))

    if frame_type in ('I', 'P'):
        decoded_Y = decode_frame(encoded, refs, BLOCK_SIZE, QUANT_SCALE)
        ref_frames.append(decoded_Y)
        if len(ref_frames) > 2:
            ref_frames.pop(0)

    bits = count_bits(encoded)
    total_bits += bits
    print(f"  Frame {frame_idx:3d} [{frame_type}]  {label}  bits: {bits:>9,}")

print(f"\n  Total bits : {total_bits:,}")
print(f"  Avg / frame: {total_bits // len(video_frames):,}")

# ── Save to disk ───────────────────────────────────────────────────────────────
for encoded, label in encoded_seq:
    out_path = os.path.join(OUTPUT_DIR, label)

    np.save(f"{out_path}_qblocks.npy", encoded['q_blocks'])

    mv = encoded['motion_vectors'] if encoded['motion_vectors'] is not None \
         else np.zeros((1, 1, 2), dtype=np.float32)
    np.save(f"{out_path}_mv.npy", mv)

    meta = {
        'type':        encoded['type'],
        'shape':       encoded['shape'],
        'hb':          encoded['hb'],
        'wb':          encoded['wb'],
        'block_size':  BLOCK_SIZE,
        'quant_scale': QUANT_SCALE,
        'gop_n':       GOP_N,
        'gop_m':       GOP_M,
    }
    np.save(f"{out_path}_meta.npy", meta, allow_pickle=True)

print(f"\n  Saved to '{OUTPUT_DIR}/'")


# ── Visualise encoded sequence ─────────────────────────────────────────────────
print("\n── Visualising encoded sequence ──")

ref_frames_vis = []

for (encoded, label), (Y_orig, _) in zip(encoded_seq, video_frames):

    frame_type = encoded['type']

    refs = ref_frames_vis[-1:] if frame_type == 'P' else \
           ref_frames_vis[-1:] * 2 if (frame_type == 'B' and ref_frames_vis) else []

    decoded_Y = decode_frame(encoded, refs, BLOCK_SIZE, QUANT_SCALE)

    if frame_type in ('I', 'P'):
        ref_frames_vis.append(decoded_Y)
        if len(ref_frames_vis) > 2:
            ref_frames_vis.pop(0)

    H, W   = decoded_Y.shape
    Y_crop = Y_orig[:H, :W]

    psnr_val = 10 * np.log10(255**2 / (np.mean((Y_crop - decoded_Y)**2) + 1e-10))
    ssim_val = ssim(Y_crop, decoded_Y, data_range=255)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].imshow(Y_crop,                     cmap='gray', vmin=0, vmax=255)
    axes[0].set_title("Original Y")
    axes[1].imshow(decoded_Y,                  cmap='gray', vmin=0, vmax=255)
    axes[1].set_title(f"Reconstructed [{frame_type}-frame]\n"
                      f"PSNR {psnr_val:.2f} dB  SSIM {ssim_val:.4f}")
    axes[2].imshow(np.abs(Y_crop - decoded_Y), cmap='hot',  vmin=0, vmax=50)
    axes[2].set_title("Error map |orig − rec|")
    for ax in axes: ax.axis('off')
    plt.suptitle(f"{label}  —  QScale={QUANT_SCALE}  Block={BLOCK_SIZE}  "
                 f"GOP N={GOP_N} M={GOP_M}")
    plt.tight_layout()
    plt.show()

    print(f"  {label}  [{frame_type}]  PSNR: {psnr_val:.2f} dB   SSIM: {ssim_val:.4f}")

In [ ]:
if DEBUG == 1: 
    # ── Parameter sweep for experiments ───────────────────────────────────────────
    experiments = [
        {'gop_n': 9,  'gop_m': 3, 'block_size': 16, 'quant_scale': 4,  'label': 'IBBP_BS16_QS4'},
        {'gop_n': 4,  'gop_m': 1, 'block_size': 16, 'quant_scale': 4,  'label': 'IPPP_BS16_QS4'},
        {'gop_n': 9,  'gop_m': 3, 'block_size': 8,  'quant_scale': 4,  'label': 'IBBP_BS8_QS4'},
        {'gop_n': 9,  'gop_m': 3, 'block_size': 16, 'quant_scale': 8,  'label': 'IBBP_BS16_QS8'},
        {'gop_n': 9,  'gop_m': 3, 'block_size': 16, 'quant_scale': 16, 'label': 'IBBP_BS16_QS16'},
    ]

    results_summary = []

    for exp in experiments:
        out_dir = f"./compressed_{exp['label']}"
        os.makedirs(out_dir, exist_ok=True)

        ref_frames = []
        total_bits = 0
        encoded_seq = []

        for frame_idx, (Y, frame_label) in enumerate(video_frames):
            frame_type = get_frame_type(frame_idx, exp['gop_n'], exp['gop_m'])

            if frame_type == 'I':
                refs = []
            elif frame_type == 'P':
                refs = [ref_frames[-1]] if ref_frames else [Y]
            else:
                prev = ref_frames[-1] if ref_frames else Y
                refs = [prev, prev]

            encoded = encode_frame(Y, frame_type, refs,
                                exp['block_size'], MIN_BLOCK,
                                ACCURACY, WINDOW_SIZE, exp['quant_scale'])
            encoded_seq.append((encoded, frame_label))

            if frame_type in ('I', 'P'):
                decoded_Y = decode_frame(encoded, refs, exp['block_size'], exp['quant_scale'])
                ref_frames.append(decoded_Y)
                if len(ref_frames) > 2:
                    ref_frames.pop(0)

            total_bits += count_bits(encoded)

        # Save to experiment-specific folder
        for encoded, frame_label in encoded_seq:
            out_path = os.path.join(out_dir, frame_label)
            np.save(f"{out_path}_qblocks.npy", encoded['q_blocks'])
            mv = encoded['motion_vectors'] if encoded['motion_vectors'] is not None \
                else np.zeros((1, 1, 2), dtype=np.float32)
            np.save(f"{out_path}_mv.npy", mv)
            meta = {
                'type': encoded['type'], 'shape': encoded['shape'],
                'hb': encoded['hb'], 'wb': encoded['wb'],
                'block_size': exp['block_size'], 'quant_scale': exp['quant_scale'],
                'gop_n': exp['gop_n'], 'gop_m': exp['gop_m'],
            }
            np.save(f"{out_path}_meta.npy", meta, allow_pickle=True)

        results_summary.append({
            'label':       exp['label'],
            'gop_n':       exp['gop_n'],
            'gop_m':       exp['gop_m'],
            'block_size':  exp['block_size'],
            'quant_scale': exp['quant_scale'],
            'total_bits':  total_bits,
            'avg_bits':    total_bits // len(video_frames),
            'encoded_seq': encoded_seq,
            'out_dir':     out_dir,
        })

        print(f"  {exp['label']:25s}  total bits: {total_bits:>12,}  avg/frame: {total_bits//len(video_frames):>8,}")

    # ── Comparison table ───────────────────────────────────────────────────────────
    print("\n" + "=" * 75)
    print(f"  {'Config':<25}  {'GOP N':>6}  {'GOP M':>6}  {'Block':>6}  {'QScale':>7}  {'Total bits':>12}  {'Avg/frame':>10}")
    print("=" * 75)
    for r in results_summary:
        print(f"  {r['label']:<25}  {r['gop_n']:>6}  {r['gop_m']:>6}  "
            f"{r['block_size']:>6}  {r['quant_scale']:>7}  "
            f"{r['total_bits']:>12,}  {r['avg_bits']:>10,}")
    print("=" * 75)

## 2.1.i) Inverse Quantization 
- Multiplies the incoming 8x8 quantized coefficient blocks back out, using the exact same base matrix and quality scale factors assigned during the compression phase.

In [ ]:
def inverse_quantisation(q_block, mode='I', quant_scale=4):
    """
    Performs inverse quantisation on a single 8x8 block.
    Uses the same base MPEG matrices and scale factor as quantise_block.

    Parameters:
        q_block     : 2D int16 array (8x8) of quantised DCT coefficients.
        mode        : 'I', 'P', or 'B' — must match what was used during encoding.
        quant_scale : Same scale factor used during quantisation.

    Returns:
        block : 2D float32 array of reconstructed DCT coefficients.
    """
    base_matrix   = QUANT_MATRIX_I if mode == 'I' else QUANT_MATRIX_PB
    scaled_matrix = base_matrix * quant_scale
    return q_block.astype(np.float32) * scaled_matrix

## 2.1.ii) Inverse DCT2 
- Converts the restored frequency-domain coefficients back into standard spatial-domain luminance and residual error intensities via an inverse 2D transform.

In [ ]:
def inverse_dct2_blocks(q_blocks, h_blocks, w_blocks):
    """
    Applies the inverse 2D DCT to every 8x8 block of quantised coefficients,
    reconstructing the spatial-domain signal (either raw Y for I-frames,
    or the residual for P/B-frames).

    Parameters:
        q_blocks  : 4D array (h_blocks, w_blocks, 8, 8) of DCT coefficients.
        h_blocks  : Number of block rows.
        w_blocks  : Number of block columns.

    Returns:
        recovered : 2D float32 array of reconstructed values.
    """
    recovered = np.zeros((h_blocks * 8, w_blocks * 8), dtype=np.float32)

    for i in range(h_blocks):
        for j in range(w_blocks):
            block = q_blocks[i, j].astype(np.float32)
            # Apply inverse DCT along columns then rows (separable 2D IDCT)
            idct_block = idct(idct(block, norm='ortho', axis=0), norm='ortho', axis=1)
            recovered[i*8:(i+1)*8, j*8:(j+1)*8] = idct_block

    return recovered

## 2.1.iii) Motion Reconstruction
- Reconstitutes full pixel values for predictive P and B frames by combining the decoded residual differences with motion-compensated blocks extracted from reference frames.

In [ ]:
def recover_motion_compensated_frame(residual, motion_vectors, ref_frames, block_size):
    """
    Recovers the full pixel data of a P or B frame by adding the decoded
    residual back onto the motion-compensated prediction.

    Parameters:
        residual       : 2D float32 array — IDCT output (the decoded residual).
        motion_vectors : Array (rows_mb, cols_mb, 2) of (dy, dx) motion vectors.
        ref_frames     : List of decoded reference Y frames.
        block_size     : Macro-block size used during encoding.

    Returns:
        decoded_Y : 2D float32 array of reconstructed luminance, clipped to [0, 255].
    """
    rows_mb, cols_mb = motion_vectors.shape[:2]
    prediction = np.zeros((rows_mb * block_size, cols_mb * block_size), dtype=np.float32)
    ref = ref_frames[0]   # primary reference frame

    for i in range(rows_mb):
        for j in range(cols_mb):
            dy, dx = motion_vectors[i, j]
            ry = int(np.clip(round(i * block_size + dy), 0, ref.shape[0] - block_size))
            rx = int(np.clip(round(j * block_size + dx), 0, ref.shape[1] - block_size))
            prediction[i*block_size:(i+1)*block_size,
                       j*block_size:(j+1)*block_size] = \
                ref[ry:ry + block_size, rx:rx + block_size]

    H_pred, W_pred = prediction.shape
    decoded_Y = prediction + residual[:H_pred, :W_pred]
    return np.clip(decoded_Y, 0, 255).astype(np.float32)

## 2.1.iv) Quality Assessment Metrics
- Quantifies mathematical and structural losses between the reconstructed video frames and the original input sequence using peak signal-to-noise ratio (PSNR) and structural similarity index (SSIM) measurements.

In [ ]:
def evaluate_quality(original, reconstructed):
    """
    Evaluates the quality of a reconstructed frame against the original.

    Parameters:
        original      : 2D float32 array — original luminance frame.
        reconstructed : 2D float32 array — decoded luminance frame.

    Returns:
        psnr : Peak Signal-to-Noise Ratio in dB (higher = better).
        ssim : Structural Similarity Index (1.0 = perfect).
    """
    # Align shapes in case of cropping during encoding
    H, W  = reconstructed.shape
    orig  = original[:H, :W]

    mse_val  = np.mean((orig - reconstructed) ** 2)
    psnr_val = 10 * np.log10(255**2 / (mse_val + 1e-10))
    ssim_val = ssim(orig, reconstructed, data_range=255)

    return psnr_val, ssim_val

## 2.2) Video Reconstruction and Parameter Impact Analysis

- Process: This master decoding script reads the compressed data files, runs the reversal pipeline, and displays the reconstructed sequence on screen.  

- Analytical Documentation: For every experimental iteration, the script pairs measured quality metrics with their exact encoding configurations. This documentation enables a direct performance comparison across varying macroblock architectures, pixel search boundaries, and quantization scales.

In [ ]:
# ── 2.2 — Decoder Script ──────────────────────────────────────────────────────

COMPRESSED_DIR = './compressed'
VIDEO_PATH     = './videos/input.mp4'

# Only load frame_XXXX labels, ignoring any leftover car1/car2 files
for exp_result in results_summary:
    COMPRESSED_DIR = exp_result['out_dir']
    print(f"\n{'='*65}")
    print(f"  Decoding: {exp_result['label']}")
    print(f"{'='*65}")
    
    
    saved_labels = sorted(
        set(f.replace('_qblocks.npy', '').replace('_mv.npy', '').replace('_meta.npy', '')
            for f in os.listdir(COMPRESSED_DIR)
            if f.endswith('.npy') and f.startswith('frame_'))
    )

    print(f"  Found {len(saved_labels)} encoded frames in '{COMPRESSED_DIR}/'")

    # Reload original frames from video into a dict keyed by label
    print("  Loading original frames from video for comparison...")
    original_frames = {}
    cap = cv2.VideoCapture(VIDEO_PATH)
    for label in saved_labels:
        frame_number = int(label.replace('frame_', ''))
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
        ret, frame_bgr = cap.read()
        if ret:
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            converted = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2YCrCb)
            original_frames[label] = cv2.split(converted)[0].astype(np.float32)
    cap.release()
    print(f"  Loaded {len(original_frames)} original frames from video")

    ref_frames_dec = []
    all_psnr       = []
    all_ssim       = []

    for label in saved_labels:
        out_path = os.path.join(COMPRESSED_DIR, label)

        # ── Load saved files ───────────────────────────────────────────────────────
        q_blocks = np.load(f"{out_path}_qblocks.npy")
        mv       = np.load(f"{out_path}_mv.npy")
        meta     = np.load(f"{out_path}_meta.npy", allow_pickle=True).item()

        frame_type  = meta['type']
        hb, wb      = meta['hb'], meta['wb']
        block_size  = meta['block_size']
        quant_scale = meta['quant_scale']
        gop_n       = meta['gop_n']
        gop_m       = meta['gop_m']

        # ── 2.1.i — Inverse quantisation ──────────────────────────────────────────
        dct_rec = np.array(
            [[inverse_quantisation(q_blocks[i, j], mode=frame_type, quant_scale=quant_scale)
            for j in range(wb)]
            for i in range(hb)],
            dtype=np.float32
        )

        # ── 2.1.ii — Inverse DCT ──────────────────────────────────────────────────
        rec_signal = inverse_dct2_blocks(dct_rec, hb, wb)

        # ── 2.1.iii — Motion compensation ─────────────────────────────────────────
        if frame_type == 'I':
            decoded_Y = np.clip(rec_signal, 0, 255).astype(np.float32)
        else:
            refs = ref_frames_dec[-1:] if ref_frames_dec else []
            if not refs:
                decoded_Y = np.clip(rec_signal, 0, 255).astype(np.float32)
            else:
                decoded_Y = recover_motion_compensated_frame(
                    rec_signal, mv, refs, block_size
                )

        if frame_type in ('I', 'P'):
            ref_frames_dec.append(decoded_Y)
            if len(ref_frames_dec) > 2:
                ref_frames_dec.pop(0)

        # ── 2.1.iv — Quality evaluation ───────────────────────────────────────────
        Y_orig             = original_frames[label]
        psnr_val, ssim_val = evaluate_quality(Y_orig, decoded_Y)
        all_psnr.append(psnr_val)
        all_ssim.append(ssim_val)

        # ── Display ───────────────────────────────────────────────────────────────
        H, W   = decoded_Y.shape
        Y_crop = Y_orig[:H, :W]

        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        axes[0].imshow(Y_crop,                     cmap='gray', vmin=0, vmax=255)
        axes[0].set_title("Original Y")
        axes[1].imshow(decoded_Y,                  cmap='gray', vmin=0, vmax=255)
        axes[1].set_title(f"Reconstructed [{frame_type}-frame]\n"
                        f"PSNR {psnr_val:.2f} dB   SSIM {ssim_val:.4f}")
        axes[2].imshow(np.abs(Y_crop - decoded_Y), cmap='hot',  vmin=0, vmax=50)
        axes[2].set_title("Error map |orig − rec|")
        for ax in axes: ax.axis('off')
        plt.suptitle(
            f"{label}  [{frame_type}]  |  "
            f"QScale={quant_scale}   Block={block_size}px   "
            f"GOP N={gop_n} M={gop_m}   "
            f"Window={WINDOW_SIZE}   Accuracy={ACCURACY}"
        )
        plt.tight_layout()
        plt.show()

        print(f"  {label:15s}  [{frame_type}]  "
            f"PSNR: {psnr_val:.2f} dB   SSIM: {ssim_val:.4f}  |  "
            f"QScale={quant_scale}  Block={block_size}  GOP N={gop_n} M={gop_m}")

    # ── Summary ───────────────────────────────────────────────────────────────────
    print("\n" + "=" * 65)
    print(f"  Frames decoded : {len(saved_labels)}")
    print(f"  Avg PSNR       : {np.mean(all_psnr):.2f} dB")
    print(f"  Avg SSIM       : {np.mean(all_ssim):.4f}")
    print(f"  Min PSNR       : {np.min(all_psnr):.2f} dB  ({saved_labels[np.argmin(all_psnr)]})")
    print(f"  Max PSNR       : {np.max(all_psnr):.2f} dB  ({saved_labels[np.argmax(all_psnr)]})")
    print("=" * 65)